In [ ]:
# Demo

# Overview
- MNIST dataset
  + Goal: estimating summation of handwritten digit images
  + Input: a set of 28x28 images
  + Output: summation of given handwritten digit images
- Defining an CNN+LSTM for MNIST dataset
- Training and evaluating an CNN+LSTM

# Import libraries

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets

import time
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# (1) Data

In [ ]:
# the data, shuffled and split between train and test sets
train_set = datasets.MNIST(root='./data', train=True, download=True)
test_set = datasets.MNIST(root='./data', train=False, download=True)
(X_train_raw, y_train_temp), (X_test_raw, y_test_temp) = (train_set.data.numpy(), train_set.targets.numpy()), \
                                                         (test_set.data.numpy(), test_set.targets.numpy())

#basic image processing
X_train_raw = X_train_raw.astype('float32')
X_test_raw  = X_test_raw.astype('float32')
X_train_raw /= 255
X_test_raw  /= 255

print('X_train_raw shape:', X_train_raw.shape)
print(X_train_raw.shape[0], 'train samples')
print(X_test_raw.shape[0], 'test samples')

## Original data
- Input: 48x48 image
- Output: label (0, 1,..., 9)

## Data preprocessing
- Read images from MNIST
- Compute their summation

## New data
- Input: eight 48x48 images
- Output: summation of eight input images

In [ ]:
n_train = X_train_raw.shape[0]
x_train = []
y_train = []

n_test = X_test_raw.shape[0]
x_test = []
y_test = []

np.random.seed(2018)
maxToAdd = 8
size = 28

# PyTorch layout: (N, time steps, channels, H, W); float32 to save memory
x_train = np.zeros((n_train, maxToAdd, 1, size, size), dtype='float32')

for i in range(0, n_train):
    #initialize a training example of max_num_time_steps, im_size, im_size
    output = np.zeros((maxToAdd, 1, size, size), dtype='float32')
    #decide how many MNIST images to put in that tensor
    numToAdd = int(np.ceil(np.random.rand()*maxToAdd))
    #sample that many images
    indices = np.random.choice(X_train_raw.shape[0], size=numToAdd)
    example = X_train_raw[indices]
    #sum up the outputs for new output
    exampleY = y_train_temp[indices]
    output[0:numToAdd,0,:,:] = example
    x_train[i, :, :, :, :] = output
    y_train.append(np.sum(exampleY))

y_train = np.array(y_train, dtype='float32')

print("x_train shape: ", x_train.shape)
print("y_train shape: ", y_train.shape)

# Test the model
x_test = np.zeros((n_test, maxToAdd, 1, size, size), dtype='float32')
for i in range(0, n_test):
    output = np.zeros((maxToAdd, 1, size, size), dtype='float32')
    numToAdd = int(np.ceil(np.random.rand()*maxToAdd))
    indices = np.random.choice(X_test_raw.shape[0], size=numToAdd)
    example = X_test_raw[indices]
    exampleY = y_test_temp[indices]
    output[0:numToAdd,0,:,:] = example
    x_test[i, :, :, :, :] = output
    y_test.append(np.sum(exampleY))

x_test = np.array(x_test)
y_test = np.array(y_test, dtype='float32')

print("x_test shape: ", x_test.shape)
print("y_test shape: ", y_test.shape)

# (2) Declare model

In [ ]:
class MnistCNNLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        # input: a sequence of maxToAdd 28x28 MNIST images, shape (N, maxToAdd, 1, 28, 28)
        # The CNN part is applied to every time step independently (Keras TimeDistributed):
        # the batch and time dimensions are merged into (N * maxToAdd, 1, 28, 28).

        # conv, pooling layers + dropout
        self.conv1 = nn.Sequential(nn.Conv2d(1, 64, kernel_size=3, padding=1), nn.ReLU())     # 'same' padding
        self.pool1 = nn.MaxPool2d(kernel_size=2)
        self.conv_dropout1 = nn.Dropout(p=0.2)

        self.conv2 = nn.Sequential(nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU())
        self.pool2 = nn.MaxPool2d(kernel_size=2)
        self.conv_dropout2 = nn.Dropout(p=0.2)

        self.conv3 = nn.Sequential(nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.ReLU())
        self.pool3 = nn.MaxPool2d(kernel_size=2)
        self.conv_dropout3 = nn.Dropout(p=0.2)

        # similar to the MLP example!
        # matrix ---> vector  (28 -> 14 -> 7 -> 3, so 256 x 3 x 3 features per time step)
        self.flatten = nn.Flatten()

        # LSTM + dropout
        self.lstm1 = nn.LSTM(input_size=256 * 3 * 3, hidden_size=128, batch_first=True)   # return_sequences=True
        self.lstm2 = nn.LSTM(input_size=128, hidden_size=128, batch_first=True)           # only last step is used
        self.lstm_dropout = nn.Dropout(p=0.22)

        # single linear output: estimated summation (regression)
        self.fc3_10ways_softmax = nn.Linear(128, 1)

        # Keras default initialization: glorot_uniform kernels, orthogonal recurrent kernels,
        # zero biases (LSTM forget-gate bias = 1, i.e. unit_forget_bias=True)
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight_ih' in name:
                        nn.init.xavier_uniform_(param)
                    elif 'weight_hh' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias_ih' in name:
                        nn.init.zeros_(param)
                        param.data[m.hidden_size:2 * m.hidden_size] = 1.0   # gate order: i, f, g, o
                    elif 'bias_hh' in name:
                        nn.init.zeros_(param)

    def forward(self, x):
        B, T = x.shape[:2]
        x = x.reshape(B * T, *x.shape[2:])                       # TimeDistributed: (B*T, C, H, W)
        x = self.conv_dropout1(self.pool1(self.conv1(x)))
        x = self.conv_dropout2(self.pool2(self.conv2(x)))
        x = self.conv_dropout3(self.pool3(self.conv3(x)))
        x = self.flatten(x)
        x = x.reshape(B, T, -1)                                  # back to (B, T, F)

        x, _ = self.lstm1(x)                                     # full sequence (B, T, 128)
        x, _ = self.lstm2(x)
        x = x[:, -1, :]                                          # return_sequences=False -> last time step
        x = self.lstm_dropout(x)
        return self.fc3_10ways_softmax(x)


# define model
model = MnistCNNLSTM().to(device)

# print model summary
print(model)
print('Total params: {:,}'.format(sum(p.numel() for p in model.parameters())))
print('Trainable params: {:,}'.format(sum(p.numel() for p in model.parameters() if p.requires_grad)))

In [ ]:
# visualize model architecture (replacement for keras model_to_dot)
print(model)

# (3) Train defined model
- Note that the training history including loss and accuracy will be save in 'history' variable
- In case your system runs out of memory (OOM), try to decrease batch size

In [ ]:
# helper functions replacing Keras' model.fit / model.evaluate / model.predict
# (regression task: only the loss is tracked, as in the Keras version which had no extra metrics)
def fit(model, train_loader, val_loader, epochs, optimizer, criterion):
    history = {'loss': [], 'val_loss': []}
    for epoch in range(epochs):
        model.train()
        total_loss, total = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
            total += xb.size(0)
        val_loss = evaluate(model, val_loader, criterion)
        history['loss'].append(total_loss / total)
        history['val_loss'].append(val_loss)
        print('Epoch %d/%d - loss: %.4f - val_loss: %.4f'
              % (epoch + 1, epochs, history['loss'][-1], val_loss))
    return history


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        total_loss += criterion(model(xb), yb).item() * xb.size(0)
        total += xb.size(0)
    return total_loss / total


@torch.no_grad()
def predict(model, x, batch_size=128):
    """Run the model on a numpy array and return the outputs as a numpy array."""
    model.eval()
    outputs = []
    for i in range(0, len(x), batch_size):
        xb = torch.as_tensor(x[i:i + batch_size], dtype=torch.float32).to(device)
        outputs.append(model(xb).cpu())
    return torch.cat(outputs).numpy()

In [ ]:
# declare learning rate, loss function, and model metric
criterion = nn.L1Loss()   # mean absolute error
lr = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

# train the model
batch_size = 128
epochs = 30

# targets have shape (N, 1) to match the model output
train_loader = DataLoader(TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train).unsqueeze(1)),
                          batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test).unsqueeze(1)),
                         batch_size=batch_size, shuffle=False)

starting_time = time.time()
history = fit(model, train_loader, test_loader, epochs, optimizer, criterion)
print('> training time is %.4f minutes' % ((time.time() - starting_time)/60))

### Training time
- Training time with GTX 1080 is about 26.6267 minutes for 30 epochs (measured with the original Keras version)
- Training time with Core i7 is around 808 minutes (~13.5 hours) for 30 epochs (measured with the original Keras version)

### Training time
- Training time with GTX 1080 is about 26.6267 minutes for 30 epochs (measured with the original Keras version)
- Training time with Core i7 is around 808 minutes (~13.5 hours) for 30 epochs (measured with the original Keras version)

In [ ]:
score = evaluate(model, test_loader, criterion)
print('Mean absolute error of testing set:', score)

In [ ]:
# select a test image randomly
random_test_index = np.random.choice(x_test.shape[0], size=1)[0]

plt.figure(figsize=(10, 5))
for indx in range(maxToAdd):
    plt.subplot(1, maxToAdd, indx + 1)
    test_img = x_test[random_test_index, indx, 0, :, :]
    plt.axis('off')
    plt.imshow(test_img, cmap='gray')

# estimate summation of testing images
est_sum = predict(model, np.expand_dims(x_test[random_test_index], axis=0))[0][0]

print('> actual summation = %d' % y_test[random_test_index])
print('> estimated summation = %.3f' % est_sum)
plt.show()